# Step 1 — EDA Slice

Goal: validate Food.com data supports our pipeline before Stage 1 filter work.

**Check:**
1. Recipe volume + key columns (`ingredients`, `tags`, `minutes`)
2. Meal intent tags (dessert, main, etc.)
3. Cook time distribution (20/30/60 min buckets)
4. Pantry match feasibility (ingredient vocabulary)
5. Constraint keyword sanity (nuts, dairy, gluten, vegan/veg)

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config.constraints import (
    ALLERGEN_KEYWORDS,
    ANIMAL_DERIVED_KEYWORDS,
    FISH_KEYWORDS,
    MEAT_KEYWORDS,
)
from src.data.loader import get_dataset_path, load_recipes

pd.set_option("display.max_colwidth", 80)

dataset_path = get_dataset_path()
print("Dataset path:", dataset_path)

recipes = load_recipes()
print("Recipes loaded:", len(recipes))
recipes.head(3)

Dataset path: /Users/yuv/.cache/kagglehub/datasets/shuyangli94/food-com-recipes-and-user-interactions/versions/2
Recipes loaded: 231637


,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"[60-minutes-or-less, time-to-make, course, main-ingredient, cuisine, prepara...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"[make a choice and proceed with recipe, depending on size of squash , cut in...",autumn is my favorite time of year to cook! this recipe \r\ncan be prepared ...,"[winter squash, mexican seasoning, mixed spice, honey, butter, olive oil, salt]",7
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"[30-minutes-or-less, time-to-make, course, main-ingredient, cuisine, prepara...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"[preheat oven to 425 degrees f, press dough into the bottom and sides of a 1...",this recipe calls for the crust to be prebaked a bit before adding ingredien...,"[prepared pizza crust, sausage patty, eggs, milk, salt and pepper, cheese]",6
2,all in the kitchen chili,112140,130,196586,2005-02-25,"[time-to-make, course, preparation, main-dish, chili, crock-pot-slow-cooker,...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"[brown ground beef in large pot, add chopped onions to ground beef when almo...",this modified version of 'mom's' chili was a hit at our 2004 christmas party...,"[ground beef, yellow onions, diced tomatoes, tomato paste, tomato soup, rote...",13


## 1. Core column sanity

In [2]:
core_cols = ["id", "name", "minutes", "ingredients", "tags", "n_ingredients"]
missing = [c for c in core_cols if c not in recipes.columns]
print("Missing core columns:", missing or "none")

summary = {
    "recipes": len(recipes),
    "median_minutes": recipes["minutes"].median(),
    "mean_ingredients": recipes["n_ingredients"].mean(),
    "empty_ingredient_rows": (recipes["ingredients"].map(len) == 0).sum(),
    "empty_tag_rows": (recipes["tags"].map(len) == 0).sum(),
}
pd.Series(summary)

Missing core columns: none


recipes                  231637.000000
median_minutes               40.000000
mean_ingredients              9.051153
empty_ingredient_rows         0.000000
empty_tag_rows                0.000000
dtype: float64

## 2. Time budget feature (`minutes`)

In [3]:
bins = [0, 20, 30, 45, 60, 120, 9999]
labels = ["<=20", "21-30", "31-45", "46-60", "61-120", ">120"]
recipes["time_bucket"] = pd.cut(recipes["minutes"], bins=bins, labels=labels, right=True)

time_counts = recipes["time_bucket"].value_counts().sort_index()
time_pct = (time_counts / len(recipes) * 100).round(1)

pd.DataFrame({"count": time_counts, "pct": time_pct})

,count,pct
time_bucket,,
<=20,60752,26.2
21-30,37207,16.1
31-45,41486,17.9
46-60,28591,12.3
61-120,36683,15.8
>120,25562,11.0


## 3. Meal intent tags (dessert / course / cuisine)

In [4]:
all_tags = recipes["tags"].explode().dropna().astype(str).str.lower()
tag_counts = all_tags.value_counts()

# Food.com uses plural tag names in many cases (e.g. desserts, not dessert)
intent_keywords = [
    "desserts", "cakes", "main-dish", "appetizers", "breakfast", "lunch",
    "dinner", "snacks", "salad", "soups", "pasta", "bread",
]
intent_hits = tag_counts[tag_counts.index.isin(intent_keywords)]
print("Intent tag hits:\n", intent_hits)

print("\nTop 25 tags overall:\n", tag_counts.head(25))

Intent tag hits:
 tags
main-dish     71786
desserts      43203
lunch         23800
appetizers    20379
breakfast     13655
pasta         12908
cakes          9653
snacks         7159
Name: count, dtype: int64

Top 25 tags overall:
 tags
preparation           230546
time-to-make          225326
course                218148
main-ingredient       170446
dietary               165091
easy                  126062
occasion              114145
cuisine                91165
low-in-something       85776
main-dish              71786
equipment              70436
60-minutes-or-less     69990
number-of-servings     58949
meat                   56042
30-minutes-or-less     55077
vegetables             53814
taste-mood             52143
4-hours-or-less        49497
north-american         48479
3-steps-or-less        44933
15-minutes-or-less     43934
low-sodium             43349
desserts               43203
low-carb               42189
healthy                40340
Name: count, dtype: int64


## 4. Pantry match feasibility (ingredient vocabulary)

In [5]:
ingredient_series = recipes["ingredients"].explode().dropna().astype(str).str.lower().str.strip()
unique_ingredients = ingredient_series.nunique()
print("Unique ingredient strings:", unique_ingredients)

print("\nTop 30 most common ingredients:")
print(ingredient_series.value_counts().head(30))

# Rough overlap demo: if user has 3 pantry items, how many recipes match all 3?
pantry_demo = ["chicken", "garlic", "onion"]
match_mask = recipes["ingredients"].map(lambda ings: all(item in ings for item in pantry_demo))
print(f"\nDemo pantry {pantry_demo} -> recipes containing all: {match_mask.sum()}")

Unique ingredient strings: 14942

Top 30 most common ingredients:
ingredients
salt                 85746
butter               54975
sugar                44535
onion                39065
water                34914
eggs                 33761
olive oil            32822
flour                26266
milk                 25786
garlic cloves        25748
pepper               22319
brown sugar          18655
garlic               18087
all-purpose flour    17659
baking powder        17504
egg                  17304
salt and pepper      15415
parmesan cheese      14807
lemon juice          14233
baking soda          14099
vegetable oil        13912
vanilla              13315
black pepper         13098
cinnamon             12560
tomatoes             11950
sour cream           11779
garlic powder        10887
vanilla extract      10271
oil                   9925
honey                 9898
Name: count, dtype: int64

Demo pantry ['chicken', 'garlic', 'onion'] -> recipes containing all: 119


## 5. Constraint keyword sanity (Stage 1 preview)

Quick substring scan on parsed ingredient lists. Final Stage 1 will use a dedicated parser + rules module.

In [6]:
def ingredient_hits(recipe_ings, keywords):
    text = " | ".join(recipe_ings)
    return [kw for kw in keywords if kw in text]


def count_recipes_with_any(keywords):
    return recipes["ingredients"].map(lambda ings: bool(ingredient_hits(ings, keywords))).sum()

allergen_hits = {
    allergen: count_recipes_with_any(list(words))
    for allergen, words in ALLERGEN_KEYWORDS.items()
}

diet_hits = {
    "contains_meat_or_fish": count_recipes_with_any(list(MEAT_KEYWORDS) + list(FISH_KEYWORDS)),
    "contains_animal_derived": count_recipes_with_any(list(ANIMAL_DERIVED_KEYWORDS)),
}

pd.DataFrame(
    {
        "recipes_with_hits": {**allergen_hits, **diet_hits},
        "pct_of_catalog": {
            k: round(v / len(recipes) * 100, 2)
            for k, v in {**allergen_hits, **diet_hits}.items()
        },
    }
)

,recipes_with_hits,pct_of_catalog
nuts,47874,20.67
dairy,144304,62.30
gluten,109736,47.37
contains_meat_or_fish,95587,41.27
contains_animal_derived,165738,71.55


In [7]:
# Spot-check a few likely edge cases
sample_checks = recipes[
    recipes["name"].str.contains("peanut|cheese|flour|chicken|vegan", case=False, na=False)
].head(8)[["id", "name", "ingredients"]]

sample_checks

,id,name,ingredients
15,63986,chicken lickin good pork chops,"[lean pork chops, flour, salt, dry mustard, garlic powder, oil, chicken rice..."
19,23850,cream of cauliflower soup vegan,"[canola oil, onion, garlic, cauliflower, potatoes, vegetable bouillon cubes,..."
21,24701,cream of spinach soup vegan,"[onion, scallion, apple juice, olive oil, spinach, fresh parsley, celery, br..."
22,83873,crispy crunchy chicken,"[boneless skinless chicken breast halves, condensed cream of chicken soup, e..."
57,32169,make that chicken dance salsa pasta,"[tomatoes, garlic, onion, button mushrooms, hot sauce, dried oregano, dried ..."
68,71635,no bake cookie crumble cheesecake,"[gelatin, milk, cream cheese, sugar, vanilla extract, miniature semisweet ch..."
75,42570,pick me up party chicken kabobs,"[boneless chicken breast, garlic, salt, cumin, paprika, thyme, olive oil, le..."
85,103948,smells like sunday chicken fricassee with meatballs,"[boneless skinless chicken thighs, all-purpose flour, salt, fresh ground bla..."


## Step 1 decisions

After running this notebook, confirm:

- [ ] `ingredients`, `tags`, `minutes` usable with low empty-row rate
- [ ] Enough recipes in `<=20` and `<=30` min buckets for time-budget demos
- [ ] Intent tags exist for dessert/main/snack filtering
- [ ] Ingredient vocabulary large enough for pantry overlap scoring
- [ ] Allergen/diet keyword hits look plausible (not 0%, not ~100%)

If all checked → proceed to **Step 2 (ingredient parser)**.